# COS30082 — Approach 2: V1 Classification Diagnostics + V2 Training

**One notebook — Run All** (use the same project environment as the original notebooks).

1. **V1 checkpoint check:** reuse existing `checkpoints/approach2_exp*_full.pt` if available; automatically train only missing V1 checkpoints. Existing V1 files are **not overwritten**.
2. **V1 classification diagnostics:** crop vs disease vs pair Top-1/Top-5, pair bias, prediction rank, and per-image unseen errors. Official test datasets here are for **diagnosis only**, not choosing V2 parameters.
3. **V2 pair-held-out split:** select 10 entire crop–disease pairs from the **original TRAIN CSV only**. Both individual crop/disease labels must remain represented in V2 training.
4. **V2 training:** four experiments, **20 epochs each (80 total)**, differential learning rates and changed regularization weights, epoch-by-epoch checkpointing/resume, and pair-held-out/seen validation.
5. **Optional final official evaluation:** **OFF by default**. Run only after you finish model selection using the held-out validation data.

**Estimated training budgets:** If all four original V1 checkpoints already exist, V1 training = 0 epochs. Otherwise each missing V1 checkpoint costs 20 epochs. V2 full mode = 4 × 20 epochs.

**Project dependencies:** `common/config.py`, `common/dataset.py`, `common/evaluate.py`, `data/DPD_Small_*.csv`, `data/DPD_Small_dataset/`, `torch`, `timm`, `pandas`, `numpy`, `PIL`.

**Important:** V2 is a set of candidate experiments, not a guaranteed accuracy improvement. It does not use official test accuracy for checkpoint or experiment selection.


## Part A — V1: Check or create missing checkpoints

In [ ]:
import gc
import json
import sys
from pathlib import Path

import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / 'common' / 'dataset.py').is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError('Open this notebook in the project root or approaches folder')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from common import config
from common.config import (
    CHECKPOINT_DIR, DEVICE, NUM_CROPS, NUM_DISEASES, RESULTS_DIR, set_seed
)
from common.dataset import get_loaders, get_pairs, get_pair_names

PAIR_TABLE = get_pairs().to(DEVICE)
PAIR_NAMES = get_pair_names()
assert len(PAIR_TABLE) == 333
assert len(PAIR_NAMES) == 333

print('Project root:', ROOT)
print('Device:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

V1_EXPERIMENTS = [
    ('exp1_encoders', 0.0, 0.0),
    ('exp2_orthogonal', 0.1, 0.0),
    ('exp3_adversarial', 0.0, 0.3),
    ('exp4_combined', 0.1, 0.3),
]
V1_HP = {
    'backbone': 'resnet50',
    'concept_dim': 512,
    'lr': 1e-4,
    'weight_decay': 1e-4,
    'epochs': 20,
    'batch_size': 64,
    'label_smoothing': 0.1,
}

# Set True ONLY if missing V1 checkpoints must be trained again
TRAIN_MISSING_V1 = False

# Set 0 if Jupyter on Windows has DataLoader worker errors
config.NUM_WORKERS = 2
V1_DIR = RESULTS_DIR / 'approach2_v1_diagnostics'
V1_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
checkpoint_paths = {
    name: CHECKPOINT_DIR / f'approach2_{name}_full.pt'
    for name, _, _ in V1_EXPERIMENTS
}
print('Checkpoint directory:', CHECKPOINT_DIR)
for name, path in checkpoint_paths.items():
    print(name, 'FOUND' if path.is_file() else 'MISSING')

In [ ]:
class GradientReversal(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -grad_output


class DisentangleNet(nn.Module):
    def __init__(self, pretrained=True, backbone='resnet50', concept_dim=512):
        super().__init__()
        self.backbone = timm.create_model(
            backbone, pretrained=pretrained, num_classes=0
        )
        width = self.backbone.num_features
        self.crop_encoder = nn.Sequential(
            nn.Linear(width, concept_dim), nn.ReLU()
        )
        self.disease_encoder = nn.Sequential(
            nn.Linear(width, concept_dim), nn.ReLU()
        )
        self.crop_head = nn.Linear(concept_dim, NUM_CROPS)
        self.disease_head = nn.Linear(concept_dim, NUM_DISEASES)
        self.disease_from_crop = nn.Linear(concept_dim, NUM_DISEASES)
        self.crop_from_disease = nn.Linear(concept_dim, NUM_CROPS)

    def forward(self, images, use_adversary=False):
        shared = self.backbone(images)
        crop_features = self.crop_encoder(shared)
        disease_features = self.disease_encoder(shared)
        logits = (
            self.crop_head(crop_features),
            self.disease_head(disease_features),
        )
        adversarial_logits = None
        if use_adversary:
            adversarial_logits = (
                self.disease_from_crop(GradientReversal.apply(crop_features)),
                self.crop_from_disease(GradientReversal.apply(disease_features)),
            )
        return logits, adversarial_logits, (crop_features, disease_features)


def orthogonal_loss(crop_features, disease_features):
    crop_features = F.normalize(
        crop_features - crop_features.mean(0), dim=0
    )
    disease_features = F.normalize(
        disease_features - disease_features.mean(0), dim=0
    )
    return (crop_features.T @ disease_features).square().mean()


@torch.no_grad()
def pair_logits(model, images):
    (crop_logits, disease_logits), _, _ = model(images.to(DEVICE))
    crop_log = F.log_softmax(crop_logits.float(), dim=1)
    disease_log = F.log_softmax(disease_logits.float(), dim=1)
    return crop_log[:, PAIR_TABLE[:, 0]] + disease_log[:, PAIR_TABLE[:, 1]]

In [ ]:
# One-notebook default: train only V1 checkpoints that are missing.
# Already-existing V1 checkpoint files will NOT be retrained or overwritten.
TRAIN_MISSING_V1 = True
print('Train missing V1 checkpoints:', TRAIN_MISSING_V1)


## 3. Optional training for missing V1 checkpoints

This cell uses original V1 loss weights, ImageNet pretrained ResNet50 and the standard team data loaders. It skips existing checkpoints. **If all checkpoints exist, it does no training.**

In [ ]:
missing = [name for name, path in checkpoint_paths.items() if not path.is_file()]

if missing and not TRAIN_MISSING_V1:
    raise FileNotFoundError(
        'Missing V1 checkpoints: ' + ', '.join(missing)
        + '\nCopy the four *_full.pt files into checkpoints/ '
        + 'or set TRAIN_MISSING_V1 = True in the setup cell'
    )

if missing:
    print('Reproducing missing V1 models:', missing)
    set_seed(42)
    starting = DisentangleNet(
        pretrained=True,
        backbone=V1_HP['backbone'],
        concept_dim=V1_HP['concept_dim'],
    )
    initial_state = {
        k: v.detach().cpu().clone()
        for k, v in starting.state_dict().items()
    }
    del starting

    from common.evaluate import val_accuracy

    for name, orth_weight, adv_weight in V1_EXPERIMENTS:
        if name not in missing:
            continue

        print('Training V1:', name)
        set_seed(42)
        train_loader, val_loader, _, _ = get_loaders(
            batch_size=V1_HP['batch_size'],
            norm='imagenet', debug=False,
        )
        model = DisentangleNet(
            pretrained=False,
            backbone=V1_HP['backbone'],
            concept_dim=V1_HP['concept_dim'],
        )
        model.load_state_dict(initial_state)
        model = model.to(DEVICE)
        criterion = nn.CrossEntropyLoss(
            label_smoothing=V1_HP['label_smoothing']
        )
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=V1_HP['lr'],
            weight_decay=V1_HP['weight_decay'],
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=V1_HP['epochs']
        )
        best_acc = -1.0
        best_epoch = 0
        history = []

        def predict(images):
            model.eval()
            return pair_logits(model, images)

        for epoch in range(V1_HP['epochs']):
            model.train()
            total = 0.0
            steps = 0
            for images, crop_ids, disease_ids, _ in train_loader:
                images = images.to(DEVICE, non_blocking=True)
                crop_ids = crop_ids.to(DEVICE, non_blocking=True)
                disease_ids = disease_ids.to(DEVICE, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                use_amp = DEVICE.type == 'cuda' and torch.cuda.is_bf16_supported()
                with torch.autocast(
                    device_type=DEVICE.type,
                    dtype=torch.bfloat16, enabled=use_amp
                ):
                    (cl, dl), adv_logits, (cf, df) = model(
                        images, use_adversary=adv_weight > 0
                    )
                    cls = criterion(cl, crop_ids) + criterion(dl, disease_ids)
                    adv = (
                        criterion(adv_logits[0], disease_ids)
                        + criterion(adv_logits[1], crop_ids)
                    ) if adv_weight else cls.new_zeros(())
                orth = (
                    orthogonal_loss(cf.float(), df.float())
                    if orth_weight else cls.new_zeros(())
                )
                loss = cls + orth_weight * orth + adv_weight * adv
                loss.backward()
                optimizer.step()
                total += loss.item()
                steps += 1

            scheduler.step()
            val = val_accuracy(predict, val_loader)
            history.append({
                'epoch': epoch + 1,
                'train_loss': total / max(steps, 1),
                'val_top1': val,
            })
            if val > best_acc:
                best_acc, best_epoch = val, epoch + 1
                torch.save({
                    'model_name': name,
                    'state_dict': {
                        k: v.detach().cpu()
                        for k, v in model.state_dict().items()
                    },
                    'hp': dict(
                        V1_HP, orthogonal_weight=orth_weight,
                        adversarial_weight=adv_weight,
                        mixed_precision=use_amp,
                    ),
                    'best_val': best_acc,
                    'best_epoch': best_epoch,
                    'run_type': 'full',
                }, checkpoint_paths[name])
            pd.DataFrame(history).to_csv(
                V1_DIR / f'{name}_reproduced_history.csv', index=False
            )
            print(f'{name} | {epoch + 1}/20 | val {val:.2f}%')

        del model, optimizer, scheduler, train_loader, val_loader
        gc.collect()
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()

print('All four V1 checkpoints are ready')

## Part B — V1: Classification diagnostics on original held-out tests

## 4. Classification diagnostics on the original seen and unseen test sets

These are **diagnostics**, not training or model selection. Predictions use the original 333 valid candidate pairs and match the original V1 score equation.

In [ ]:
# The original team test loaders use deterministic transforms and no shuffle
_, _, seen_loader, unseen_loader = get_loaders(
    batch_size=64, norm='imagenet', debug=False
)
assert len(seen_loader.dataset) == 2000
assert len(unseen_loader.dataset) == 1448

@torch.inference_mode()
def diagnose(model, loader):
    model.eval()
    rows = []
    offset = 0

    for images, true_crop, true_disease, true_pair in loader:
        images = images.to(DEVICE, non_blocking=True)
        true_crop = true_crop.to(DEVICE)
        true_disease = true_disease.to(DEVICE)
        true_pair = true_pair.to(DEVICE)

        assert torch.equal(PAIR_TABLE[true_pair, 0], true_crop)
        assert torch.equal(PAIR_TABLE[true_pair, 1], true_disease)

        (crop_logits, disease_logits), _, _ = model(images)
        crop_log = F.log_softmax(crop_logits.float(), dim=1)
        disease_log = F.log_softmax(disease_logits.float(), dim=1)
        pred_crop = crop_log.argmax(dim=1)
        pred_disease = disease_log.argmax(dim=1)
        scores = (
            crop_log[:, PAIR_TABLE[:, 0]]
            + disease_log[:, PAIR_TABLE[:, 1]]
        )
        pred_pair = scores.argmax(dim=1)
        top5 = scores.topk(5, dim=1).indices
        true_score = scores.gather(1, true_pair[:, None]).squeeze(1)
        rank = 1 + (scores > true_score[:, None]).sum(dim=1)
        best_seen = scores[:, :config.NUM_SEEN_PAIRS].max(1).values
        best_unseen = scores[:, config.NUM_SEEN_PAIRS:].max(1).values
        margin = best_seen - best_unseen

        file_names = loader.dataset.frame['image'].iloc[
            offset:offset + len(images)
        ].tolist()
        offset += len(images)

        for i, filename in enumerate(file_names):
            tc = int(true_crop[i].item())
            td = int(true_disease[i].item())
            tp = int(true_pair[i].item())
            pc = int(pred_crop[i].item())
            pdisease = int(pred_disease[i].item())
            pp = int(pred_pair[i].item())
            pair_c = int(PAIR_TABLE[pp, 0].item())
            pair_d = int(PAIR_TABLE[pp, 1].item())
            rows.append({
                'image': filename,
                'true_pair_id': tp,
                'true_pair': PAIR_NAMES[tp],
                'predicted_pair_id': pp,
                'predicted_pair': PAIR_NAMES[pp],
                'true_crop_id': tc,
                'predicted_crop_id': pc,
                'true_disease_id': td,
                'predicted_disease_id': pdisease,
                'crop_correct': pc == tc,
                'disease_correct': pdisease == td,
                'both_correct': pc == tc and pdisease == td,
                'pair_prediction_crop_correct': pair_c == tc,
                'pair_prediction_disease_correct': pair_d == td,
                'pair_top1_correct': pp == tp,
                'pair_top5_correct': tp in top5[i].tolist(),
                'predicted_seen_pair': pp < config.NUM_SEEN_PAIRS,
                'true_pair_rank': int(rank[i].item()),
                'seen_minus_unseen_score': float(margin[i].item()),
            })

    result = pd.DataFrame(rows)
    assert (result.loc[result['both_correct'], 'pair_top1_correct']).all(), (
        'Inconsistent pair score or label mapping detected'
    )
    return result


def percent(column):
    return round(float(column.mean()) * 100, 2)


summaries, unseen_records = [], []
for name, _, _ in V1_EXPERIMENTS:
    print('Diagnosing', name)
    ckpt = torch.load(
        checkpoint_paths[name], map_location='cpu', weights_only=True
    )
    hp = ckpt.get('hp', {})
    model = DisentangleNet(
        pretrained=False,
        backbone=hp.get('backbone', 'resnet50'),
        concept_dim=int(hp.get('concept_dim', 512)),
    )
    model.load_state_dict(ckpt['state_dict'], strict=True)
    model = model.to(DEVICE).eval()

    for split, loader in [('seen', seen_loader), ('unseen', unseen_loader)]:
        detail = diagnose(model, loader)
        summaries.append({
            'experiment': name,
            'split': split,
            'num_images': len(detail),
            'crop_top1': percent(detail['crop_correct']),
            'disease_top1': percent(detail['disease_correct']),
            'both_correct': percent(detail['both_correct']),
            'pair_top1': percent(detail['pair_top1_correct']),
            'pair_top5': percent(detail['pair_top5_correct']),
            'pair_selected_crop_top1': percent(
                detail['pair_prediction_crop_correct']
            ),
            'pair_selected_disease_top1': percent(
                detail['pair_prediction_disease_correct']
            ),
            'predicted_seen_pct': percent(detail['predicted_seen_pair']),
            'mean_true_pair_rank': round(float(detail['true_pair_rank'].mean()), 2),
            'mean_seen_minus_unseen_score': round(
                float(detail['seen_minus_unseen_score'].mean()), 4
            ),
        })
        if split == 'unseen':
            detail.insert(0, 'experiment', name)
            unseen_records.append(detail)

    del model, ckpt
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

summary = pd.DataFrame(summaries)
predictions = pd.concat(unseen_records, ignore_index=True)
summary_path = V1_DIR / 'classification_summary.csv'
prediction_path = V1_DIR / 'unseen_predictions.csv'
summary.to_csv(summary_path, index=False)
predictions.to_csv(prediction_path, index=False)

print('\nV1 diagnostic summary')
display(summary)
print('\nFiles saved:')
print(summary_path)
print(prediction_path)

## 5. Interpret the diagnostics

- If **unseen disease accuracy** is low, focus on the disease feature/heads rather than immediately changing pair scores
- If both individual concepts are correct, current pair scoring must also be correct; the notebook asserts this
- `predicted_seen_pct` on the **unseen** split directly checks seen-pair preference
- These official-test findings may motivate hypotheses, but must **not** be used to select V2 learning rates, loss weights or calibration constants. Use the new pair-held-out validation in V2 for those decisions

## Part C — V2: Controlled training with Pair-Held-Out Validation

*V2 keeps the same `DisentangleNet` class defined above; it does **not** redefine or modify the V1 architecture.* The following separate V2 settings do not change saved V1 checkpoints.

**Run All performs all four V2 experiments, unless already completed and valid resumable epoch files are present.**

- Exp 1: Encoders only (`orth=0`, `adv=0`)
- Exp 2: Orthogonality (`orth=0.5`, `adv=0`)
- Exp 3: Adversarial (`orth=0`, `adv=0.1`)
- Exp 4: Combined (`orth=0.5`, `adv=0.1`)
- Batch size: 64; epochs: 20 each; backbone LR: `1e-5`; new layers LR: `1e-4`.


In [ ]:
from collections import Counter
import numpy as np
from torch.utils.data import DataLoader
from common.dataset import PlantDataset, get_transforms, load_split

DEBUG = False
SEED = 42
N_HELDOUT_PAIRS = 10
MIN_IMAGES_PER_HELDOUT_PAIR = 5
VAL_SEEN_FRACTION = 0.10
NUM_WORKERS = 2
config.NUM_WORKERS = NUM_WORKERS
BATCH_SIZE = 16 if DEBUG else 64
EPOCHS = 1 if DEBUG else 20

# These LR values are candidate settings, not known best values
BACKBONE_LR = 1e-5
NEW_LAYERS_LR = 1e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
CONCEPT_DIM = 512
USE_AMP = DEVICE.type == 'cuda' and torch.cuda.is_bf16_supported()

# Reuse saved epoch state if AI Lab session was interrupted
RESUME_IF_POSSIBLE = True

V2_EXPERIMENTS = [
    ('exp1_encoders', 0.0, 0.0),
    ('exp2_orthogonal', 0.5, 0.0),
    ('exp3_adversarial', 0.0, 0.1),
    ('exp4_combined', 0.5, 0.1),
]

RUN_MODE = 'debug' if DEBUG else 'full'
V2_RESULTS = RESULTS_DIR / 'approach2_v2' / RUN_MODE
V2_CHECKPOINTS = CHECKPOINT_DIR / 'approach2_v2' / RUN_MODE
V2_RESULTS.mkdir(parents=True, exist_ok=True)
V2_CHECKPOINTS.mkdir(parents=True, exist_ok=True)

print('Mode:', RUN_MODE)
print('Epochs per experiment:', EPOCHS)
print('Batch size:', BATCH_SIZE)
print('Mixed precision:', USE_AMP)
print('Results folder:', V2_RESULTS)
print('Checkpoints folder:', V2_CHECKPOINTS)

## 2. Create fixed pair-held-out validation

All images of 10 selected training pairs are **removed from training**. Each held-out pair retains its crop ID and disease ID somewhere in the remaining training pairs. Another 10% image-level seen validation is sampled from remaining training pairs. The official test CSVs are never used to choose training samples or hyperparameters.

In [ ]:
source = load_split(config.TRAIN_CSV)
assert len(source) == 8000, 'Expected the 8,000-image DPD-Small train CSV'

unique_pairs = source.drop_duplicates('pair_row').set_index('pair_row')
pair_counts = source.groupby('pair_row').size()

# Keep exactly the same held-out pair IDs for future reruns
heldout_path = RESULTS_DIR / 'approach2_v2' / 'heldout_pairs_seed42.csv'
heldout_path.parent.mkdir(parents=True, exist_ok=True)

if heldout_path.is_file():
    heldout_pair_rows = pd.read_csv(heldout_path)['pair_row'].astype(int).tolist()
    print('Reusing existing held-out pair selection')
else:
    remaining_crop_pairs = Counter(unique_pairs['crop'].astype(int).tolist())
    remaining_disease_pairs = Counter(unique_pairs['disease'].astype(int).tolist())
    eligible = pair_counts[pair_counts >= MIN_IMAGES_PER_HELDOUT_PAIR].index.tolist()
    rng = np.random.default_rng(SEED)
    eligible = rng.permutation(eligible).tolist()
    heldout_pair_rows = []

    for pair_id in eligible:
        crop_id = int(unique_pairs.loc[pair_id, 'crop'])
        disease_id = int(unique_pairs.loc[pair_id, 'disease'])
        if remaining_crop_pairs[crop_id] > 1 and remaining_disease_pairs[disease_id] > 1:
            heldout_pair_rows.append(int(pair_id))
            remaining_crop_pairs[crop_id] -= 1
            remaining_disease_pairs[disease_id] -= 1
        if len(heldout_pair_rows) == N_HELDOUT_PAIRS:
            break

    if len(heldout_pair_rows) != N_HELDOUT_PAIRS:
        raise RuntimeError(
            f'Only found {len(heldout_pair_rows)} valid held-out pairs; '
            'review N_HELDOUT_PAIRS and MIN_IMAGES_PER_HELDOUT_PAIR'
        )
    pd.DataFrame({'pair_row': heldout_pair_rows}).to_csv(
        heldout_path, index=False
    )

assert len(heldout_pair_rows) == N_HELDOUT_PAIRS
assert len(set(heldout_pair_rows)) == N_HELDOUT_PAIRS
heldout_frame = source[source['pair_row'].isin(heldout_pair_rows)].copy()
remaining = source[~source['pair_row'].isin(heldout_pair_rows)].copy()

seen_val = remaining.groupby('pair_row', group_keys=False).sample(
    frac=VAL_SEEN_FRACTION, random_state=SEED
)
train_frame = remaining.drop(index=seen_val.index).copy()

assert not set(train_frame['pair_row']) & set(heldout_pair_rows)
assert not set(seen_val['pair_row']) & set(heldout_pair_rows)
assert set(heldout_frame['crop']) <= set(train_frame['crop'])
assert set(heldout_frame['disease']) <= set(train_frame['disease'])
assert set(train_frame.index).isdisjoint(set(seen_val.index))
assert len(train_frame) + len(seen_val) + len(heldout_frame) == len(source)

# Available training pairs are candidate classes, not the original 323 seen pairs
train_pair_mask = torch.zeros(len(PAIR_TABLE), dtype=torch.bool, device=DEVICE)
train_pair_mask[torch.tensor(train_frame['pair_row'].unique(), device=DEVICE)] = True

split_stats = {
    'seed': SEED,
    'train_images': len(train_frame),
    'seen_validation_images': len(seen_val),
    'pair_heldout_validation_images': len(heldout_frame),
    'training_pair_count': int(train_frame['pair_row'].nunique()),
    'heldout_pair_rows': heldout_pair_rows,
    'heldout_pair_count': len(heldout_pair_rows),
}
(V2_RESULTS / 'split_summary.json').write_text(
    json.dumps(split_stats, indent=2)
)
heldout_desc = unique_pairs.loc[heldout_pair_rows, [
    'crop', 'disease', 'pair_name'
]].copy()
heldout_desc['num_images'] = pair_counts.loc[heldout_pair_rows].values
heldout_desc.to_csv(V2_RESULTS / 'heldout_pair_details.csv')

print(json.dumps(split_stats, indent=2))
display(heldout_desc)

## 3. Data loaders and model

In [ ]:
train_transform, eval_transform = get_transforms('imagenet')


def subset_for_debug(frame, n):
    return frame.sample(n=min(n, len(frame)), random_state=SEED)


def make_v2_loaders():
    tr = subset_for_debug(train_frame, 200) if DEBUG else train_frame
    seen = subset_for_debug(seen_val, 80) if DEBUG else seen_val
    held = subset_for_debug(heldout_frame, 80) if DEBUG else heldout_frame

    def loader(frame, transform, shuffle):
        return DataLoader(
            PlantDataset(frame, transform, 'imagenet'),
            batch_size=BATCH_SIZE,
            shuffle=shuffle,
            num_workers=NUM_WORKERS,
            pin_memory=(DEVICE.type == 'cuda'),
            persistent_workers=(NUM_WORKERS > 0),
            generator=torch.Generator().manual_seed(SEED),
        )

    return (
        loader(tr, train_transform, True),
        loader(seen, eval_transform, False),
        loader(held, eval_transform, False),
    )

print('Data loaders prepared')

### Shared V1/V2 architecture

Both use the *same* model class defined in Part A (`ResNet50 → crop/disease 512-dimensional encoders → crop/disease heads`). This intentionally allows the V1 and V2 results to be compared at the architecture level, while varying V2 training and loss settings.

## 4. V2 validation metrics

Pair Top-1 and Top-5 always score all **333 candidate pairs**. The held-out validation pairs are part of that candidate list but never occur in V2 training. Model selection uses the harmonic mean of seen-validation and held-out-pair Top-1.

In [ ]:
@torch.inference_mode()
def evaluate_loader(model, loader):
    model.eval()
    n = crop_ok = disease_ok = both_ok = top1_ok = top5_ok = trained_pair_predictions = 0
    for images, crop_labels, disease_labels, true_pair in loader:
        images = images.to(DEVICE, non_blocking=True)
        crop_labels = crop_labels.to(DEVICE)
        disease_labels = disease_labels.to(DEVICE)
        true_pair = true_pair.to(DEVICE)
        (cl, dl), _, _ = model(images)
        crop_pred = cl.argmax(1)
        disease_pred = dl.argmax(1)
        crop_log = F.log_softmax(cl.float(), dim=1)
        disease_log = F.log_softmax(dl.float(), dim=1)
        scores = crop_log[:, PAIR_TABLE[:, 0]] + disease_log[:, PAIR_TABLE[:, 1]]
        pair_pred = scores.argmax(1)
        pair_top5 = scores.topk(5, dim=1).indices
        n += len(images)
        cc = crop_pred == crop_labels
        dc = disease_pred == disease_labels
        crop_ok += int(cc.sum())
        disease_ok += int(dc.sum())
        both_ok += int((cc & dc).sum())
        top1_ok += int((pair_pred == true_pair).sum())
        top5_ok += int((pair_top5 == true_pair[:, None]).any(1).sum())
        trained_pair_predictions += int(train_pair_mask[pair_pred].sum())
    return {
        'images': n,
        'crop_top1': round(100 * crop_ok / n, 2),
        'disease_top1': round(100 * disease_ok / n, 2),
        'both_correct': round(100 * both_ok / n, 2),
        'pair_top1': round(100 * top1_ok / n, 2),
        'pair_top5': round(100 * top5_ok / n, 2),
        'predicted_training_pair_pct': round(100 * trained_pair_predictions / n, 2),
    }


def harmonic_mean(a, b):
    return 0.0 if a + b == 0 else round(2 * a * b / (a + b), 2)


@torch.inference_mode()
def measure_feature_correlation(model, loader):
    model.eval()
    crop_features, disease_features = [], []
    for images, _, _, _ in loader:
        _, _, (crop, disease) = model(images.to(DEVICE))
        crop_features.append(crop.float().cpu())
        disease_features.append(disease.float().cpu())
    crop_features = torch.cat(crop_features)
    disease_features = torch.cat(disease_features)
    return round(float(orthogonal_loss(crop_features, disease_features)), 6)


def get_optimizer(model):
    return torch.optim.AdamW([
        {'params': model.backbone.parameters(), 'lr': BACKBONE_LR},
        {'params': list(model.crop_encoder.parameters())
                   + list(model.disease_encoder.parameters())
                   + list(model.crop_head.parameters())
                   + list(model.disease_head.parameters())
                   + list(model.disease_from_crop.parameters())
                   + list(model.crop_from_disease.parameters()),
         'lr': NEW_LAYERS_LR},
    ], weight_decay=WEIGHT_DECAY)


def train_one_epoch(model, train_loader, optimizer, criterion, orth_weight, adv_weight):
    model.train()
    totals = {'total_loss': 0.0, 'classification_loss': 0.0,
              'orthogonality_loss': 0.0, 'adversarial_loss': 0.0}
    count = 0
    for images, crops, diseases, _ in train_loader:
        images = images.to(DEVICE, non_blocking=True)
        crops = crops.to(DEVICE)
        diseases = diseases.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(
            device_type=DEVICE.type, dtype=torch.bfloat16, enabled=USE_AMP
        ):
            (cl, dl), adversary, (cf, df) = model(
                images, use_adversary=(adv_weight > 0)
            )
            cls = criterion(cl, crops) + criterion(dl, diseases)
            adv = (
                criterion(adversary[0], diseases)
                + criterion(adversary[1], crops)
            ) if adv_weight else cls.new_zeros(())
        orth = (
            orthogonal_loss(cf.float(), df.float())
            if orth_weight else cls.new_zeros(())
        )
        loss = cls + orth_weight * orth + adv_weight * adv
        if not torch.isfinite(loss).item():
            raise FloatingPointError('Non-finite training loss')
        loss.backward()
        optimizer.step()
        bs = len(images)
        count += bs
        for k, value in [
            ('total_loss', loss), ('classification_loss', cls),
            ('orthogonality_loss', orth), ('adversarial_loss', adv)
        ]:
            totals[k] += float(value.detach()) * bs
    return {k: round(v / count, 6) for k, v in totals.items()}

## 5. Train all four V2 experiments

Identical pretrained ResNet50 initialisation, train/validation split, seed, optimiser family, epochs, and batch size. Only the two loss coefficients vary among the four experiments. Each epoch writes history, the best checkpoint and a resumable last checkpoint. Training does **not** use the official unseen test.

In [ ]:
set_seed(SEED)
starting_model = DisentangleNet(
    pretrained=True, backbone='resnet50', concept_dim=CONCEPT_DIM
)
initial_state = {
    k: value.detach().cpu().clone()
    for k, value in starting_model.state_dict().items()
}
del starting_model
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()

summary_records = []
all_histories = []

for experiment_name, orth_weight, adv_weight in V2_EXPERIMENTS:
    print('\n' + '=' * 60)
    print('Experiment:', experiment_name)
    set_seed(SEED)
    train_loader, seen_loader, heldout_loader = make_v2_loaders()
    model = DisentangleNet(
        pretrained=False, backbone='resnet50', concept_dim=CONCEPT_DIM
    )
    model.load_state_dict(initial_state, strict=True)
    model = model.to(DEVICE)

    optimizer = get_optimizer(model)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=EPOCHS
    )
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    best_file = V2_CHECKPOINTS / f'{experiment_name}_best.pt'
    last_file = V2_CHECKPOINTS / f'{experiment_name}_last.pt'
    history_file = V2_RESULTS / f'{experiment_name}_history.csv'
    signature = {
        'experiment': experiment_name,
        'mode': RUN_MODE,
        'seed': SEED,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'backbone_lr': BACKBONE_LR,
        'new_layers_lr': NEW_LAYERS_LR,
        'weight_decay': WEIGHT_DECAY,
        'label_smoothing': LABEL_SMOOTHING,
        'concept_dim': CONCEPT_DIM,
        'orthogonal_weight': orth_weight,
        'adversarial_weight': adv_weight,
        'heldout_pair_rows': heldout_pair_rows,
    }

    start_epoch = 0
    best_key = (-1.0, -1.0, -1.0)
    best_epoch = 0
    history = []
    if last_file.is_file() and RESUME_IF_POSSIBLE:
        last = torch.load(last_file, map_location='cpu', weights_only=True)
        if last['signature'] != signature:
            raise RuntimeError(
                f'Existing {last_file.name} belongs to different settings. '
                'Use a different output folder or deliberately back up and remove that file'
            )
        model.load_state_dict(last['model_state'], strict=True)
        optimizer.load_state_dict(last['optimizer_state'])
        scheduler.load_state_dict(last['scheduler_state'])
        start_epoch = int(last['epoch'])
        best_key = tuple(last['best_key'])
        best_epoch = int(last['best_epoch'])
        history = last['history']
        print('Resuming after epoch:', start_epoch)
        del last

    for epoch in range(start_epoch, EPOCHS):
        losses = train_one_epoch(
            model, train_loader, optimizer, criterion,
            orth_weight, adv_weight
        )
        scheduler.step()
        seen_metrics = evaluate_loader(model, seen_loader)
        held_metrics = evaluate_loader(model, heldout_loader)
        hm = harmonic_mean(
            seen_metrics['pair_top1'], held_metrics['pair_top1']
        )
        record = {
            'experiment': experiment_name,
            'epoch': epoch + 1,
            **losses,
            'seen_val_top1': seen_metrics['pair_top1'],
            'seen_val_top5': seen_metrics['pair_top5'],
            'heldout_top1': held_metrics['pair_top1'],
            'heldout_top5': held_metrics['pair_top5'],
            'heldout_crop_top1': held_metrics['crop_top1'],
            'heldout_disease_top1': held_metrics['disease_top1'],
            'heldout_both_correct': held_metrics['both_correct'],
            'heldout_pred_training_pair_pct': held_metrics['predicted_training_pair_pct'],
            'val_harmonic_mean': hm,
        }
        history.append(record)
        current_key = (
            hm, held_metrics['pair_top1'], seen_metrics['pair_top1']
        )

        if current_key > best_key:
            best_key, best_epoch = current_key, epoch + 1
            torch.save({
                'state_dict': {
                    k: v.detach().cpu().clone()
                    for k, v in model.state_dict().items()
                },
                'hp': signature,
                'best_epoch': best_epoch,
                'seen_validation': seen_metrics,
                'heldout_validation': held_metrics,
                'harmonic_mean': hm,
            }, best_file)

        pd.DataFrame(history).to_csv(history_file, index=False)
        torch.save({
            'signature': signature,
            'epoch': epoch + 1,
            'model_state': {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            },
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'best_key': best_key,
            'best_epoch': best_epoch,
            'history': history,
        }, last_file)
        print(
            f'{experiment_name} | {epoch + 1}/{EPOCHS} | '
            f'Seen val {seen_metrics["pair_top1"]:.2f}% | '
            f'Held-out {held_metrics["pair_top1"]:.2f}% | '
            f'H {hm:.2f}%'
        )

    best_checkpoint = torch.load(
        best_file, map_location='cpu', weights_only=True
    )
    bseen = best_checkpoint['seen_validation']
    bheld = best_checkpoint['heldout_validation']
    model.load_state_dict(best_checkpoint['state_dict'])
    heldout_correlation = measure_feature_correlation(model, heldout_loader)
    summary_records.append({
        'experiment': experiment_name,
        'orthogonal_weight': orth_weight,
        'adversarial_weight': adv_weight,
        'best_epoch': best_checkpoint['best_epoch'],
        'seen_val_top1': bseen['pair_top1'],
        'heldout_top1': bheld['pair_top1'],
        'heldout_top5': bheld['pair_top5'],
        'heldout_crop_top1': bheld['crop_top1'],
        'heldout_disease_top1': bheld['disease_top1'],
        'heldout_pred_training_pair_pct': bheld['predicted_training_pair_pct'],
        'heldout_feature_correlation': heldout_correlation,
        'val_harmonic_mean': best_checkpoint['harmonic_mean'],
        'checkpoint': str(best_file),
    })
    all_histories.extend(history)
    summary_df = pd.DataFrame(summary_records).sort_values(
        ['val_harmonic_mean', 'heldout_top1', 'seen_val_top1'],
        ascending=False,
    )
    summary_df.to_csv(V2_RESULTS / 'validation_summary.csv', index=False)
    pd.DataFrame(all_histories).to_csv(
        V2_RESULTS / 'training_history.csv', index=False
    )
    del model, optimizer, scheduler, train_loader, seen_loader, heldout_loader
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

print('\nV2 validation comparison')
display(summary_df)
print('All outputs are in:', V2_RESULTS)

## 6. Explain results and lock the candidate

- **Primary research metric:** Held-out-pair Top-1, alongside harmonic mean and seen-validation Top-1
- `heldout_pred_training_pair_pct` indicates preference for combinations available in V2 training
- Comparing V2 *held-out* scores directly with V1 *official unseen* scores is **not valid**: different images/splits
- V2 tuning must use the fixed held-out split only; the previous official unseen test was already consulted in V1
- If you later want a team-comparable official test result, lock one V2 configuration first and run the optional final stage below **once**, without tuning further from its official test score

## 7. Optional: retrain one locked V2 configuration on the normal training split and evaluate official tests

**OFF by default.** When satisfied with development validation, change `RUN_FINAL_OFFICIAL_TEST=True`, rerun this *last cell only* after prior cells completed. It trains **one new selected V2 model for 20 epochs** using the original team 90/10 seen train/validation split, saves best seen-validation epoch, then evaluates original seen/unseen tests once. Because the official test has already been examined during V1, treat this as exploratory / not a pristine untouched test for further tuning.

In [ ]:
RUN_FINAL_OFFICIAL_TEST = False

if not RUN_FINAL_OFFICIAL_TEST:
    print('Official test skipped (recommended during V2 development)')
else:
    from common.evaluate import evaluate, val_accuracy
    if DEBUG:
        raise RuntimeError('Set DEBUG=False for final training')
    chosen = summary_df.iloc[0]['experiment']
    coefficients = {
        n: (o, a) for n, o, a in V2_EXPERIMENTS
    }
    orth_weight, adv_weight = coefficients[chosen]
    print('Locked experiment before official test:', chosen)
    print('Re-training using normal original train split')

    set_seed(SEED)
    final_model = DisentangleNet(
        pretrained=False, backbone='resnet50', concept_dim=CONCEPT_DIM
    )
    final_model.load_state_dict(initial_state)
    final_model = final_model.to(DEVICE)
    final_optimizer = get_optimizer(final_model)
    final_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        final_optimizer, T_max=20
    )
    final_criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    final_train, final_val, final_seen, final_unseen = get_loaders(
        batch_size=64, norm='imagenet', debug=False
    )
    final_best = -1.0
    final_best_path = V2_CHECKPOINTS / 'locked_final_best.pt'
    final_log = []

    def final_predict(images):
        final_model.eval()
        return pair_logits(final_model, images)

    for epoch in range(20):
        losses = train_one_epoch(
            final_model, final_train, final_optimizer,
            final_criterion, orth_weight, adv_weight
        )
        final_scheduler.step()
        seen_validation_top1 = val_accuracy(final_predict, final_val)
        final_log.append({
            'epoch': epoch + 1, 'seen_validation_top1': seen_validation_top1,
            **losses,
        })
        pd.DataFrame(final_log).to_csv(
            V2_RESULTS / 'locked_final_history.csv', index=False
        )
        if seen_validation_top1 > final_best:
            final_best = seen_validation_top1
            torch.save({
                'selected_experiment': chosen,
                'best_epoch': epoch + 1,
                'seen_validation_top1': seen_validation_top1,
                'state_dict': {
                    k: v.detach().cpu().clone()
                    for k, v in final_model.state_dict().items()
                },
            }, final_best_path)
        print('Final', epoch + 1, '/20', 'seen val', seen_validation_top1)

    locked = torch.load(final_best_path, map_location='cpu', weights_only=True)
    final_model.load_state_dict(locked['state_dict'])
    final_model.eval()
    report = evaluate(
        final_predict, final_seen, final_unseen,
        'approach2_v2_locked_final',
        {'chosen': chosen, 'backbone_lr': BACKBONE_LR,
         'new_layers_lr': NEW_LAYERS_LR,
         'orthogonal_weight': orth_weight,
         'adversarial_weight': adv_weight,
         'epochs': 20, 'selected_from': 'pair_heldout_validation'},
        robust=False,
    )
    (V2_RESULTS / 'locked_final_official_results.json').write_text(
        json.dumps(report, indent=2)
    )
    print('Official results saved:', V2_RESULTS / 'locked_final_official_results.json')

## Outputs to collect

**V1 diagnostic CSVs** (existing V1 model weights reused whenever possible):

- `results/approach2_v1_diagnostics/classification_summary.csv`
- `results/approach2_v1_diagnostics/unseen_predictions.csv`

**V2 validation files** (the official unseen test is *not used* for V2 model selection):

- `results/approach2_v2/full/validation_summary.csv`
- `results/approach2_v2/full/training_history.csv`
- `results/approach2_v2/full/heldout_pair_details.csv`
- `results/approach2_v2/full/split_summary.json`
- `checkpoints/approach2_v2/full/{experiment}_best.pt` and `{experiment}_last.pt`

**Practical AI Lab run:** Open the notebook from your repository folder in VS Code, select the existing `.venv` Python kernel, set `DEBUG = False` (already the default), then click **Run All**. Do not change settings halfway through a run. If interrupted, rerun: V2's completed epoch state is reused when its signature matches the current settings.
